# Fast learners (Project 17)

**How to use**
- **Run the experiment:** pick `RUN_MODE` in the Config cell, then *Run all*. At the end, the results are saved to a `results_<mode>_<date>.json` file.
- **Only analyse saved results (no GPU needed):** upload the `.json` file, set `LOAD_RESULTS` in the Config cell, run **Config → Setup → Load results**, then everything below *Results*.

# **Install**

In [ ]:
import importlib.util
if importlib.util.find_spec("optuna") is None:      # Colab: install; already installed on Rocket
    %pip install -q --upgrade jax-ai-stack optuna

# **Config**
The only cell you need to change.

In [ ]:
import os

# test = a few minutes, quick = ~15 min, full = the real experiment (~3 h on a GPU)
RUN_MODE = os.environ.get("RUN_MODE", "test")     # on Rocket the job script sets RUN_MODE=full

# Analysis only: path to a saved results file. Then run Config -> Setup -> "Load results" -> Results
LOAD_RESULTS = None                               # e.g. "results_full_20261031-1200.json"

ALL_BUDGETS = [2_560, 12_800, 32_000, 64_000, 128_000]   # training images per candidate during search
RUN_SETTINGS = {
    "test":  dict(budgets=[2_560],     seeds=[0],             generations=2,  epochs=2,  top_k=3, recheck_seeds=1),
    "quick": dict(budgets=ALL_BUDGETS, seeds=[0, 1],          generations=5,  epochs=2,  top_k=5, recheck_seeds=3),
    "full":  dict(budgets=ALL_BUDGETS, seeds=[0, 1, 2, 3, 4], generations=20, epochs=20, top_k=5, recheck_seeds=3),
}
settings = RUN_SETTINGS[RUN_MODE]
BUDGETS       = settings["budgets"]
SEEDS         = settings["seeds"]           # independent search runs per (budget, method)
GENERATIONS   = settings["generations"]     # CPG * GENERATIONS = networks per search
EPOCHS        = settings["epochs"]          # length of the final retraining (1 epoch = 50,000 images)
TOP_K         = settings["top_k"]           # re-check the TOP_K best of each search ...
RECHECK_SEEDS = settings["recheck_seeds"]   # ... on this many new seeds; best average = winner

# Same for every mode
SEARCH_METHODS = ["GA", "MAP-Elites", "Optuna TPE", "Random search"]
CPG = 24                                    # candidates per generation
FINAL_SEEDS = 3                             # retraining seeds per winner / baseline
EVAL_EVERY = 640                            # validation accuracy every 640 images
THRESHOLD = [0.75, 0.80, 0.85]              # speed = images needed to reach these accuracies

print(f"RUN_MODE={RUN_MODE}: budgets {BUDGETS}, seeds {SEEDS}, {GENERATIONS} generations x {CPG}, {EPOCHS} epochs")

# **Setup**

In [ ]:
import os, time, json, pickle
import jax, jax.numpy as jnp
import numpy as np
import optuna
import optax
import matplotlib.pyplot as plt

BATCH_SIZES = [32, 64, 128]

N_TRAIN = 50_000
SPLIT_SEED = 42
DATA_SEED = 1234
SGD_MOMENTUM = 0.9

# Search space - genes and their ranges
SEARCH_SPACE = {
    "lr": ("log", 1e-4, 1.0),
    "optimizer": ("choice", ["sgd", "adam"]),
    "batch_size": ("choice", BATCH_SIZES),
    "layers": ("choice", [1, 2, 3]),                    # number of hidden Dense layers
    "width": ("int", 16, 256),                          # units per hidden layer
    "activation": ("choice", ["relu", "tanh"]),
}

# Baseline models
BASELINES = {
    "Adam 1e-3": dict(lr=1e-3, optimizer="adam", batch_size=128, layers=2, width=128, activation="relu"),
    "Adam 3e-4": dict(lr=3e-4, optimizer="adam", batch_size=128, layers=2, width=128, activation="relu"),
    "SGD 1e-2":  dict(lr=1e-2, optimizer="sgd",  batch_size=128, layers=2, width=128, activation="relu"),
}
N_GENES = len(SEARCH_SPACE)

def decode(genome):
    """Genome -> decode -> Hyperparameters"""
    g = np.clip(np.asarray(genome, dtype=float), 0.0, 1.0)
    hp = {}
    for x, (name, spec) in zip(g, SEARCH_SPACE.items()):
        kind = spec[0]
        if kind == "log":
            lo, hi = np.log10(spec[1]), np.log10(spec[2])
            hp[name] = float(10 ** (lo + x * (hi - lo)))
        elif kind == "int":
            hp[name] = int(round(spec[1] + x * (spec[2] - spec[1])))
        else:
            options = spec[1]
            hp[name] = options[min(int(x * len(options)), len(options) - 1)]
    return hp

def encode(hp):
    """Hyperparameters -> encode -> Genome"""
    g = []
    for name, spec in SEARCH_SPACE.items():
        kind, v = spec[0], hp[name]
        if kind == "log":
            lo, hi = np.log10(spec[1]), np.log10(spec[2])
            g.append((np.log10(v) - lo) / (hi - lo))
        elif kind == "int":
            g.append((v - spec[1]) / (spec[2] - spec[1]))
        else:
            options = spec[1]
            g.append((options.index(v) + 0.5) / len(options))
    return np.array(g)

def describe(hp):
    return (f"{hp['optimizer']:<4} lr={hp['lr']:.4f} batch={hp['batch_size']:3d} "
            f"{hp['layers']}x{hp['width']:3d} {hp['activation']}")

def n_params(hp, n_in=784, n_out=10):
    w, L = hp["width"], hp["layers"]
    return n_in * w + w + (L - 1) * (w * w + w) + w * n_out + n_out



# MAP-Elites (Learning rate * Number of network parameters)
GRID_BINS = 8
GOOD_CELL_MARGIN = 0.03
_P_MIN = n_params(dict(width=16, layers=1))
_P_MAX = n_params(dict(width=256, layers=3))

def grid_cell(hp):
    """Return the (row, col) cell of the diversity grid for one network."""
    lr_pos = (np.log10(hp["lr"]) + 4) / 4  # 0..1
    size_pos = (np.log10(n_params(hp)) - np.log10(_P_MIN)) / (np.log10(_P_MAX) - np.log10(_P_MIN))
    to_bin = lambda p: int(np.clip(p * GRID_BINS, 0, GRID_BINS - 1))
    return to_bin(lr_pos), to_bin(size_pos)

def sample_order(n_samples, seed=DATA_SEED):
    """Order of training images: identical for every network. Reshuffles each epoch,
    so budgets larger than 50,000 images work."""
    rng = np.random.default_rng(seed)
    n_epochs = -(-n_samples // N_TRAIN)
    return np.concatenate([rng.permutation(N_TRAIN) for _ in range(n_epochs)])[:n_samples]

print("JAX backend:", jax.default_backend(), "| devices:", jax.devices())

# **Load Data**

In [ ]:
#Loading data
from tensorflow.keras.datasets import fashion_mnist

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Normalize images to [0, 1]
x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

# Create a fixed 50k/10k training-validation split
rng = np.random.default_rng(42)
indices = rng.permutation(len(x_train))

train_idx = indices[:50000]
val_idx = indices[50000:]

x_val, y_val = x_train[val_idx], y_train[val_idx]
x_train, y_train = x_train[train_idx], y_train[train_idx]

# Flatten 28x28 images
x_train = jnp.asarray(x_train.reshape(-1, 784))
x_val = jnp.asarray(x_val.reshape(-1, 784))
x_test = jnp.asarray(x_test.reshape(-1, 784))

y_train = jnp.asarray(y_train)
y_val = jnp.asarray(y_val)
y_test = jnp.asarray(y_test)

print("Training:", x_train.shape)
print("Validation:", x_val.shape)
print("Test:", x_test.shape)


In [ ]:
#Show image
plt.imshow(x_train[0].reshape(28, 28), cmap="gray")
plt.title(f"Label: {y_train[0]}")
plt.axis("off")
plt.show()

# **Training**

In [ ]:
# TRAINING
MAX_WIDTH = SEARCH_SPACE["width"][2]
GROUP = 8

def init_params(key, width):
    """Initialize the weights and biases of the neural network."""

    # Create a separate random key for each layer
    keys = jax.random.split(key, 4)

    # Initialize weights with random values scaled by the input size
    def init_weights(key, input_size, output_size, fan_in):
        weights = jax.random.normal(key, (input_size, output_size))
        return weights / jnp.sqrt(fan_in)

    # Create weights and biases for each layer
    params = {
        "W1": init_weights(keys[0], 784, MAX_WIDTH, 784),
        "b1": jnp.zeros(MAX_WIDTH),

        "W2": init_weights(keys[1], MAX_WIDTH, MAX_WIDTH, width),
        "b2": jnp.zeros(MAX_WIDTH),

        "W3": init_weights(keys[2], MAX_WIDTH, MAX_WIDTH, width),
        "b3": jnp.zeros(MAX_WIDTH),

        "Wo": init_weights(keys[3], MAX_WIDTH, 10, width),
        "bo": jnp.zeros(10)
    }

    return params

def forward(params, config, images):
    """Run the images through the neural network."""

    # Mask neurons that are outside the selected layer width
    mask = (jnp.arange(MAX_WIDTH) < config["width"]).astype(jnp.float32)

    # Choosactivation
    def activate(values):
        return jnp.where(
            config["tanh"],
            jnp.tanh(values),
            jax.nn.relu(values)
        )

    # First hidden layer
    hidden = activate(images @ params["W1"] + params["b1"])
    hidden = hidden * mask

    # Second hidden layer
    second_layer = activate(hidden @ params["W2"] + params["b2"])
    second_layer = second_layer * mask
    hidden = jnp.where(config["layers"] >= 2, second_layer, hidden)

    # Third hidden layer
    third_layer = activate(hidden @ params["W3"] + params["b3"])
    third_layer = third_layer * mask
    hidden = jnp.where(config["layers"] >= 3, third_layer, hidden)

    # Output layer produces scores for the 10 clothing categories
    output = hidden @ params["Wo"] + params["bo"]

    return output

def accuracy(p, cfg, x, y):
    return jnp.mean(jnp.argmax(forward(p, cfg, x), axis=1) == y)

def train_net(config, key, batches, train_images, train_labels, val_images, val_labels, test_images, test_labels):
    """Train one neural network and evaluate its accuracy."""

    # network's weights and biases
    params = init_params(key, config["width"])

    # optimizers
    adam = optax.adam(config["lr"])
    sgd = optax.sgd(config["lr"], momentum=SGD_MOMENTUM)
    adam_state = adam.init(params)
    sgd_state = sgd.init(params)

    # the loss
    def loss_fn(params, images, labels):
        predictions = forward(params, config, images)
        return optax.softmax_cross_entropy_with_integer_labels(predictions, labels).mean()

    # Train on one batch
    def train_step(carry, batch_indices):
        params, adam_state, sgd_state = carry

        images = train_images[batch_indices]
        labels = train_labels[batch_indices]

        gradients = jax.grad(loss_fn)(params, images, labels)

        adam_updates, adam_state = adam.update(gradients, adam_state, params)
        sgd_updates, sgd_state = sgd.update(gradients, sgd_state, params)

        # Select the correct optimizer
        updates = jax.tree_util.tree_map(
            lambda a, b: jnp.where(config["adam"], a, b),
            adam_updates, sgd_updates
        )

        params = optax.apply_updates(params, updates)
        return (params, adam_state, sgd_state), None

    # Train on all batches and evaluate after each group
    def train_group(carry, batch_group):
        carry, _ = jax.lax.scan(train_step, carry, batch_group)
        params = carry[0]
        val_accuracy = accuracy(params, config, val_images, val_labels)
        return carry, val_accuracy

    # Store validation accuracy during training
    initial_state = (params, adam_state, sgd_state)
    final_state, learning_curve = jax.lax.scan(train_group, initial_state, batches)
    params = final_state[0]

    # Evaluate the final network
    test_accuracy = accuracy(params, config, test_images, test_labels)

    return learning_curve, test_accuracy

# vmap = train many networks at once; jit = compile. in_axes: 0 = differs per network, None = shared
train_many = jax.jit(jax.vmap(train_net, in_axes=(0, 0, None, None, None, None, None, None, None)))

def train_population(hps, budget, seed, eval_every=None):
    """Train multiple networks in parallel and return their accuracies."""

    # Prepare the training budget and image order
    eval_every = eval_every or budget
    budget = (budget // eval_every) * eval_every

    #different training seeds change the network initialization but not the image order
    image_order = sample_order(budget)
    keys = jax.random.split(jax.random.PRNGKey(seed), len(hps))

    # Store results
    val = np.zeros(len(hps))
    test = np.zeros(len(hps))
    curves = [None] * len(hps)

    #use the same training batches
    batch_sizes = sorted({hp["batch_size"] for hp in hps})

    for batch_size in batch_sizes:

        batches = jnp.asarray(image_order.reshape(budget // eval_every, eval_every // batch_size,batch_size))
        # networks that use this batch size
        network_ids = [i for i, hp in enumerate(hps) if hp["batch_size"] == batch_size]
        # Train networks in smaller groups to avoid GPU memory issues
        for start in range(0, len(network_ids), CPG):
            group_ids = network_ids[start:start + CPG]
            # Pad the group to a multiple of GROUP for parallel training
            group_size = ((len(group_ids) + GROUP - 1) // GROUP) * GROUP
            group_size = min(CPG, group_size)
            padded_ids = group_ids + [group_ids[-1]] * (group_size - len(group_ids))
            # Convert hyperparameters into JAX arrays
            config = {
              "lr": jnp.array([hps[i]["lr"] for i in padded_ids], dtype=jnp.float32),
              "adam": jnp.array([hps[i]["optimizer"] == "adam" for i in padded_ids]),
              "layers": jnp.array([hps[i]["layers"] for i in padded_ids]),
              "width": jnp.array([hps[i]["width"] for i in padded_ids]),
              "tanh": jnp.array([hps[i]["activation"] == "tanh" for i in padded_ids])
            }

            # Train the group of networks in parallel
            group_curves, group_test = train_many(config, keys[jnp.array(padded_ids)], batches, x_train, y_train, x_val, y_val, x_test, y_test)
            # Convert JAX results to NumPy
            group_curves = np.asarray(group_curves)
            group_test = np.asarray(group_test)
            # Save results in the original network order
            for position, network_id in enumerate(group_ids):
                curves[network_id] = [float(accuracy) for accuracy in group_curves[position]]
                val[network_id] = group_curves[position, -1]
                test[network_id] = group_test[position]

    return val, curves, test

def train_one(hp, budget, seed, eval_every=None):
    """Train one network. Returns (final val accuracy, learning curve)."""
    val, curves, _ = train_population([hp], budget, seed, eval_every)
    return float(val[0]), (curves[0] if eval_every else [])

def fitness(hps, budget, seed):
    """FITNESS = validation accuracy after `budget` training images, for each network."""
    return train_population(hps, budget, seed)[0]

In [ ]:
# Qucik model check
hp = BASELINES["Adam 1e-3"]
acc, curve = train_one(hp, budget=12_800, seed=0, eval_every=EVAL_EVERY)
samples = np.arange(1, len(curve) + 1) * EVAL_EVERY
plt.plot(samples, curve, marker="o"); plt.grid(True)
plt.xlabel("Training samples seen"); plt.ylabel("Validation accuracy")
plt.title(f"Baseline: {describe(hp)}"); plt.show()
print(f"Final validation accuracy: {acc:.2%}")

# **Search implementation**

In [ ]:
def mutate(genomes, rng, sigma=0.1):
    """add small random noise to every gene"""
    return np.clip(genomes + sigma * rng.normal(size=genomes.shape), 0.0, 1.0)

def crossover(a, b, rng):
    """Uniform crossover: every gene comes from parent a or parent b with 50% chance."""
    return np.where(rng.random(a.shape) < 0.5, a, b)

class RandomSearch:
    """Baseline search: random genomes."""
    def __init__(self, seed):
        self.rng = np.random.default_rng(seed)

    def propose(self, n):
        return self.rng.random((n, N_GENES))

    def update(self, genomes, fit):
        pass

class GA:
    """Genetic algorithm: parent selection based on accuracy, crossover and mutation"""
    def __init__(self, seed):
      self.rng = np.random.default_rng(seed)
      self.pop = None # Population
      self.fit = None # Fitness

    def propose(self, n):
      # First generation is random
      if self.pop is None:
        return self.rng.random((n, N_GENES))

      # Accuracy based selection
      w = self.fit - self.fit.min() + 1e-6
      p = w / w.sum()
      parents_a = self.pop[self.rng.choice(len(self.pop), n, p=p)]
      parents_b = self.pop[self.rng.choice(len(self.pop), n, p=p)]
      return mutate(crossover(parents_a, parents_b, self.rng), self.rng)

    def update(self, genomes, fit):
      if self.pop is None:
        self.pop, self.fit = np.array(genomes), np.array(fit)
        return

      # Competition between parent and children
      all_g = np.concatenate([self.pop, genomes])
      all_f = np.concatenate([self.fit, fit])

      # Keep the best candidates
      keep = np.argsort(all_f)[::-1][:len(genomes)]
      self.pop, self.fit = all_g[keep], all_f[keep]

class MAPElites:
    """Keeps the best network in every cell of the grid. New candidates are mutated children of random best networks."""
    def __init__(self, seed):
      self.rng = np.random.default_rng(seed)
      self.archive = {}

    def propose(self, n):
      # First generation is random
      if not self.archive:
            return self.rng.random((n, N_GENES))

      elites = np.array([g for g, f in self.archive.values()])
      parents_a = elites[self.rng.integers(len(elites), size=n)]
      parents_b = elites[self.rng.integers(len(elites), size=n)]
      return mutate(crossover(parents_a, parents_b, self.rng), self.rng)

    def update(self, genomes, fit):
      for g, f in zip(genomes, fit):
        cell = grid_cell(decode(g))
        if cell not in self.archive or f > self.archive[cell][1]:
          self.archive[cell] = (g, f)

class OptunaTPE:
    """A model based optimizer. Suggests genomes that its model thinks will score well."""
    def __init__(self, seed):
      self.study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
      self.trials = []

    def propose(self, n):
      self.trials = [self.study.ask() for _ in range(n)]
      return np.array([[t.suggest_float(f"g{i}", 0.0, 1.0) for i in range(N_GENES)]for t in self.trials])

    def update(self, genomes, fit):
      for t, f in zip(self.trials, fit):
            self.study.tell(t, float(np.nan_to_num(f)))

SEARCH_CLASSES = {"GA": GA, "MAP-Elites": MAPElites, "Optuna TPE": OptunaTPE, "Random search": RandomSearch}

# **Experimental run**

In [ ]:
def run_search(method, budget, seed):
    """Shared loop for all four search methods."""
    all_genomes, all_fitness = [], []
    for gen in range(GENERATIONS):
        # picks 24 candidates
        genomes = method.propose(CPG)
        hps = [decode(g) for g in genomes]

        # train + measure
        fit = fitness(hps, budget, seed)

         # learn from results
        method.update(genomes, fit)
        all_genomes += list(genomes); all_fitness += list(fit)
    return np.array(all_genomes), np.array(all_fitness)

runs = []
t0 = time.time()
for budget in BUDGETS:
    for name, Method in SEARCH_CLASSES.items():
        for seed in SEEDS:
            t = time.time()
            G, F = run_search(Method(seed), budget, seed)

            # re-check the TOP_K best with new seeds; the best average is the winner
            top = G[np.argsort(F)[::-1][:TOP_K]]
            top_hps = [decode(g) for g in top]
            recheck = np.mean([fitness(top_hps, budget, seed=1000 + 10 * seed + r)
                               for r in range(RECHECK_SEEDS)], axis=0)
            winner = top[np.argmax(recheck)]

            runs.append(dict(budget=budget, method=name, seed=seed, genomes=G, fitness=F,
                             winner=winner, winner_recheck=float(recheck.max())))
            print(f"budget={budget:>7} {name:<14} seed {seed}: best {F.max():.4f} "
                  f"rechecked {recheck.max():.4f} [{time.time() - t:.0f}s]  {describe(decode(winner))}")
print(f"All searches done in {time.time() - t0:.0f}s")

# **Retrain winners**

In [ ]:
# RETRAIN WINNER

# Networks to retrain: the baselines + one winner from every search run
networks = []
for name, hp in BASELINES.items():
    networks.append({"name": "Baseline " + name, "method": "Baseline", "budget": None, "hp": hp})
for run in runs:
    name = f"{run['method']} B={run['budget']} s{run['seed']}"
    networks.append({"name": name, "method": run["method"], "budget": run["budget"], "hp": decode(run["winner"])})

all_hps = [net["hp"] for net in networks]
all_curves = []
all_test_accuracies = []

t0 = time.time()
for s in range(FINAL_SEEDS):
    _, curves, test_accuracies = train_population(all_hps, EPOCHS * N_TRAIN, seed=500 + s, eval_every=EVAL_EVERY)
    all_curves.append(curves)
    all_test_accuracies.append(test_accuracies)
    print(f"Seed {s + 1}/{FINAL_SEEDS} done [{time.time() - t0:.0f}s]")

# Store individual learning curves and their average
for i, net in enumerate(networks):
    net["seed_curves"] = np.array([all_curves[s][i] for s in range(FINAL_SEEDS)])
    net["curve"] = net["seed_curves"].mean(axis=0)

    net["test_accuracy"] = np.mean([all_test_accuracies[s][i] for s in range(FINAL_SEEDS)])

# **Save results**
One file with everything the analysis needs: all searches (`runs`), the retrained networks (`networks`) and the settings used. Download it from the file panel to share.

In [ ]:
results_file = f"results_{RUN_MODE}_{time.strftime('%Y%m%d-%H%M')}.json"
saved_settings = dict(RUN_MODE=RUN_MODE, BUDGETS=BUDGETS, SEEDS=SEEDS, GENERATIONS=GENERATIONS, EPOCHS=EPOCHS,
                      TOP_K=TOP_K, RECHECK_SEEDS=RECHECK_SEEDS, SEARCH_METHODS=SEARCH_METHODS, CPG=CPG,
                      FINAL_SEEDS=FINAL_SEEDS, EVAL_EVERY=EVAL_EVERY, THRESHOLD=THRESHOLD,
                      SEARCH_SPACE=SEARCH_SPACE, BASELINES=BASELINES)

def to_plain(obj):
    """numpy -> plain Python, so the file opens anywhere (JSON, any numpy version)."""
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    raise TypeError(type(obj))

with open(results_file, "w") as f:
    json.dump(dict(runs=runs, networks=networks, settings=saved_settings), f, default=to_plain)
print(f"Saved {results_file} ({os.path.getsize(results_file) / 1e6:.1f} MB)")

# **Load results**
Only needed for analysing a saved file: set `LOAD_RESULTS` in the Config cell, run Config and Setup, then this cell.

In [ ]:
if LOAD_RESULTS:
    with open(LOAD_RESULTS) as f:
        saved = json.load(f)
    runs, networks, s = saved["runs"], saved["networks"], saved["settings"]
    for run in runs:
        for key in ("genomes", "fitness", "winner"):
            run[key] = np.array(run[key])
    for net in networks:
        net["seed_curves"], net["curve"] = np.array(net["seed_curves"]), np.array(net["curve"])

    # use the settings of the saved run, not the ones in the Config cell
    RUN_MODE, BUDGETS, SEEDS, GENERATIONS, EPOCHS = s["RUN_MODE"], s["BUDGETS"], s["SEEDS"], s["GENERATIONS"], s["EPOCHS"]
    SEARCH_METHODS, CPG, EVAL_EVERY, THRESHOLD = s["SEARCH_METHODS"], s["CPG"], s["EVAL_EVERY"], s["THRESHOLD"]
    print(f"Loaded {LOAD_RESULTS}: mode {RUN_MODE}, {len(runs)} searches, {len(networks)} retrained networks")
else:
    print("LOAD_RESULTS is None -> using the results from this session")

# **Results**

## Overview
Speed = training images a network needs to reach 75 / 80 / 85% validation accuracy (averaged over the retraining seeds, `never` = at least one seed never got there). Final = after the full retraining.

In [ ]:
import pandas as pd

def images_to_reach(curve, threshold):
    """Images seen when the curve first reaches the threshold (inf = never)."""
    for i, acc in enumerate(curve):
        if acc >= threshold:
            return (i + 1) * EVAL_EVERY
    return np.inf

for net in networks:
    net["speed"] = {th: np.mean([images_to_reach(c, th) for c in net["seed_curves"]]) for th in THRESHOLD}
    net["final_val"] = net["seed_curves"][:, -1].mean()

baselines = [net for net in networks if net["method"] == "Baseline"]
winners = [net for net in networks if net["method"] != "Baseline"]
images = np.arange(1, len(networks[0]["curve"]) + 1) * EVAL_EVERY     # x-axis of every learning curve

# One row per network
overview = pd.DataFrame([{
    "network": net["name"],
    **{f"to {th:.0%}": net["speed"][th] for th in THRESHOLD},
    "final val": net["final_val"],
    "test": net["test_accuracy"],
    "hyperparameters": describe(net["hp"]),
} for net in networks])
display(overview.round(4).replace(np.inf, "never"))

In [ ]:
# Averages per search budget and method (the winners of all seeds together)
summary = pd.DataFrame([{
    "budget": net["budget"],
    "method": net["method"],
    **{f"to {th:.0%}": net["speed"][th] for th in THRESHOLD},
    "final val": net["final_val"],
    "test": net["test_accuracy"],
} for net in winners])
display(summary.groupby(["budget", "method"]).mean().round(4).replace(np.inf, "never"))

## H1: do short-budget searches find fast learners?

In [ ]:
# How many winners reach each accuracy with fewer images than the BEST baseline?
for th in THRESHOLD:
    best_baseline = min(b["speed"][th] for b in baselines)
    print(f"\nReach {th:.0%}: best baseline needs {best_baseline:,.0f} images")
    for budget in BUDGETS:
        counts = []
        for method in SEARCH_METHODS:
            group = [w for w in winners if w["budget"] == budget and w["method"] == method]
            faster = sum(w["speed"][th] < best_baseline for w in group)
            counts.append(f"{method} {faster}/{len(group)}")
        print(f"  budget {budget:>7,}:  " + "  |  ".join(counts))

In [ ]:
budget_colors = plt.cm.viridis(np.linspace(0, 0.9, len(BUDGETS)))
baseline_styles = ["--", ":", "-."]
first_epoch = images <= N_TRAIN

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: average learning curve of the winners of each budget, first epoch only
for budget, color in zip(BUDGETS, budget_colors):
    mean_curve = np.mean([w["curve"] for w in winners if w["budget"] == budget], axis=0)
    ax1.plot(images[first_epoch], mean_curve[first_epoch], color=color, linewidth=2.5,
             label=f"Winners, budget {budget:,}")
for b, style in zip(baselines, baseline_styles):
    ax1.plot(images[first_epoch], b["curve"][first_epoch], color="gray", linestyle=style, linewidth=2, label=b["name"])
for th in THRESHOLD:
    ax1.axhline(th, color="black", linewidth=0.8)
ax1.set(xscale="log", ylim=(0.5, 0.9), xlabel="Training images seen (log scale)", ylabel="Validation accuracy",
        title="Learning curves in the first epoch")
ax1.legend(fontsize=8, loc="lower right")

# Right: images needed to reach 80%, one dot per network
for b, marker in zip(baselines, ["s", "^", "D"]):
    ax2.scatter(0, b["speed"][0.80], color="gray", marker=marker, s=80, label=b["name"])
for i, (budget, color) in enumerate(zip(BUDGETS, budget_colors), start=1):
    speeds = [w["speed"][0.80] for w in winners if w["budget"] == budget and np.isfinite(w["speed"][0.80])]
    ax2.scatter(i + np.random.uniform(-0.15, 0.15, len(speeds)), speeds, color=color)
ax2.set_xticks(range(len(BUDGETS) + 1), ["Baselines"] + [f"Winners\nbudget {b:,}" for b in BUDGETS])
ax2.set(ylim=(0, None), ylabel="Images needed to reach 80% (lower = faster)", title="Speed to 80% (one dot = one network)")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig("h1_results.png", dpi=150)
plt.show()

## H2: do fast learners pay in final accuracy?

In [ ]:
# Final test accuracy of the winners: rows = search budget, columns = method
print("Baselines:", ", ".join(f"{b['name']} {b['test_accuracy']:.4f}" for b in baselines))
display(summary.pivot_table(index="budget", columns="method", values="test").round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for budget, color in zip(BUDGETS, budget_colors):
    group = [w for w in winners if w["budget"] == budget and np.isfinite(w["speed"][0.80])]
    ax.scatter([w["speed"][0.80] for w in group], [w["test_accuracy"] for w in group],
               color=color, s=60, label=f"Winners, budget {budget:,}")
for b, marker in zip(baselines, ["s", "^", "D"]):
    ax.scatter(b["speed"][0.80], b["test_accuracy"], color="black", marker=marker, s=90, label=b["name"])
ax.set(xlabel="Images needed to reach 80% (lower = faster)", ylabel="Final test accuracy",
       title="Speed vs final accuracy (top-left = fast and good)")
ax.legend(fontsize=8)
ax.grid(True)
plt.show()

## Search methods vs random search
Accuracy of each search's winner (re-checked on fresh seeds), compared within the same budget and seed.

In [ ]:
from scipy.stats import wilcoxon

search_df = pd.DataFrame([{"budget": r["budget"], "method": r["method"], "seed": r["seed"],
                           "winner accuracy": r["winner_recheck"]} for r in runs])
comparison = search_df.pivot(index=["budget", "seed"], columns="method", values="winner accuracy")
display(comparison.groupby("budget").mean().round(4))

# One-sided Wilcoxon: is the method better than random search? (needs >= 5 seeds to ever reach p < 0.05)
for budget in BUDGETS:
    data = comparison.loc[budget]
    for method in [m for m in SEARCH_METHODS if m != "Random search"]:
        differences = data[method] - data["Random search"]
        p_value = 1.0 if np.allclose(differences, 0) else wilcoxon(differences, alternative="greater").pvalue
        print(f"Budget {budget:>7,}  {method:<12} > Random search:  p = {p_value:.4f} {'*' if p_value < 0.05 else ''}")

## Evolution: how the searches improve over generations

In [ ]:
EVO_BUDGET, EVO_SEED = BUDGETS[0], SEEDS[0]       # which search runs to show

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
generations = range(1, GENERATIONS + 1)
for run in runs:
    if run["budget"] != EVO_BUDGET or run["seed"] != EVO_SEED:
        continue
    scores = run["fitness"].reshape(GENERATIONS, CPG)          # rows = generations
    ax1.plot(generations, scores.mean(axis=1), marker="o", label=run["method"])
    ax2.plot(generations, np.maximum.accumulate(scores.max(axis=1)), marker="o", label=run["method"])
ax1.set(xlabel="Generation", ylabel="Average validation accuracy", title=f"Average per generation (budget {EVO_BUDGET:,}, seed {EVO_SEED})")
ax2.set(xlabel="Generation", ylabel="Best validation accuracy so far", title="Best network found so far")
for ax in (ax1, ax2):
    ax.set_xticks(list(generations))
    ax.legend()
    ax.grid(True)
plt.tight_layout()
plt.show()

## H1b: diversity (MAP-Elites vs the others)
All networks a search trained are put into the same learning-rate x network-size grid. Coverage = share of cells used, QD-score = sum of the best accuracy in every used cell.

In [ ]:
def diversity_grid(run):
    """Best validation accuracy in every grid cell (nan = no network there)."""
    grid = np.full((GRID_BINS, GRID_BINS), np.nan)
    for genome, score in zip(run["genomes"], run["fitness"]):
        row, col = grid_cell(decode(genome))
        if np.isnan(grid[row, col]) or score > grid[row, col]:
            grid[row, col] = score
    return grid

diversity_df = pd.DataFrame([{"budget": r["budget"], "method": r["method"], "seed": r["seed"],
                              "coverage": np.mean(~np.isnan(diversity_grid(r))),
                              "QD-score": np.nansum(diversity_grid(r))} for r in runs])
display(diversity_df.groupby(["budget", "method"])[["coverage", "QD-score"]].mean().unstack("method").round(3))

In [ ]:
# The grids of one search per method side by side
fig, axes = plt.subplots(1, len(SEARCH_METHODS), figsize=(18, 4.5))
for ax, method in zip(axes, SEARCH_METHODS):
    run = next(r for r in runs if r["method"] == method and r["budget"] == EVO_BUDGET and r["seed"] == EVO_SEED)
    image = ax.imshow(diversity_grid(run), origin="lower", vmin=0.1, vmax=0.9)
    ax.set(title=method, xlabel="Network size", ylabel="Learning rate")
fig.colorbar(image, ax=axes, label="Best validation accuracy in cell")
plt.show()

# Coverage over generations
for method in SEARCH_METHODS:
    run = next(r for r in runs if r["method"] == method and r["budget"] == EVO_BUDGET and r["seed"] == EVO_SEED)
    used_cells, coverage = set(), []
    for genome in run["genomes"]:
        used_cells.add(grid_cell(decode(genome)))
        coverage.append(len(used_cells) / GRID_BINS**2)
    plt.plot(range(1, GENERATIONS + 1), np.array(coverage)[CPG - 1::CPG], marker="o", label=method)
plt.xlabel("Generation")
plt.ylabel("Grid coverage")
plt.title(f"Grid coverage over generations (budget {EVO_BUDGET:,}, seed {EVO_SEED})")
plt.legend()
plt.grid(True)
plt.show()